# Extend 7B's has_error=1 stratum from 350 to 650 -- match 3B's sample size

Ninth notebook. Not driven by a power gap -- 7B's has_error=1 stratum is
already confirmed at n=350 (0.834 [0.768, 0.891], 38 wrong, clears the
registered minimum of 30 and the 0.70 threshold). This exists purely so the
two model sizes are reported on the same total n in a paper table, after a
free check already confirmed matching n wouldn't change the number: scoring
3B's own combined CSV restricted to just the 350 items 7B saw gives 0.858
[0.768, 0.924], consistent with both 3B's full-650 result (0.854) and 7B's
0.834. This run makes that consistency explicit at n=650 for 7B too, rather
than resting on 3B's side of the comparison alone.

## Design: a second, disjoint extension round

7B has already used two item ranges from the seed=42 has_error=1 ordering:
the reference run's first 150 (`skip=0`), then notebook 06's extra 200
(`skip=150, n_extra=200`), for 350 total. This notebook draws
**300 more, `skip=350, n_extra=300`** -- the next slice in the same ordering,
disjoint from both prior draws by the same construction as every prior
extension (`pilot.data.load_fermat_extra_error_items`). 350 + 300 = 650,
matching 3B's total exactly, and (since 3B's extension used the identical
seed and skip=150 ordering) **item-for-item the same 650 has_error=1 items
3B was tested on** -- not just the same count.

**Prerequisite for running:** cells 2-3 (install, auth) must run this
session; model load is 7B (heavier than notebook 08's 3B), same
OOM/4-bit-fallback pattern as notebook 06.


In [ ]:
# Install cell: GPU-dependent packages only.
%pip install -q transformers accelerate qwen-vl-utils datasets huggingface_hub


In [ ]:
# Auth & code/results access cell. Identical to notebooks 06/07/08 -- reuses
# the HF/GitHub tokens already cached on Drive from prior sessions.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


In [ ]:
# Model load cell. 7B, same OOM/4-bit fallback pattern as notebook 06 -- a
# quantized load is a genuinely different measurement, recorded loudly via
# QUANTIZED rather than silently substituted.
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
QUANTIZED = False

try:
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    print(f"Loaded {MODEL_ID} in bfloat16 (full precision).")
except torch.cuda.OutOfMemoryError:
    print(f"bfloat16 load of {MODEL_ID} did not fit -- falling back to 4-bit "
          "quantization. This changes what is being measured; the saved "
          "results record QUANTIZED=True so this is never silently glossed over.")
    from transformers import BitsAndBytesConfig

    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        device_map="auto",
        cache_dir=DRIVE_MODEL_CACHE,
    )
    QUANTIZED = True

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)

if torch.cuda.is_available():
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gib:.1f} GiB), "
          f"quantized={QUANTIZED}")


In [ ]:
# Round-2 extra-items sample cell. skip=350 -- past BOTH the reference run's
# 150 and notebook 06's extra 200, so this is disjoint from everything 7B
# has been tested on so far, and (same seed=42 ordering as 3B's own
# extension) lands on exactly 3B's items 350-649.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

SEED = 42
SKIP = 350           # has_error=1 items 7B has already covered (150 + 200)
N_EXTRA = 300         # 350 + 300 = 650, matching 3B's total exactly

extra_sample = pilot.data.load_fermat_extra_error_items(
    n_extra=N_EXTRA, seed=SEED, skip=SKIP
)
N_EXTRA = len(extra_sample)  # may shrink if the pool ran short -- keep in sync
print(f"{N_EXTRA} additional has_error=1 items drawn (items {SKIP} to {SKIP + N_EXTRA - 1} "
      f"in the seed={SEED} error-item ordering, disjoint from the 350 7B has already used)")


In [ ]:
# Grading generation for the round-2 extra items. Same K=5, same batch-
# backoff ladder as every prior grading notebook. Checkpoint is a distinct
# file (skip=350 in the name) so it can never collide with or silently
# resume the wrong slice.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
extra2_grading_path = (f"{CHECKPOINT_DIR}/grading_7b_extra_error_k{K_GRADING}_{model_slug}"
                       f"_n{N_EXTRA}_skip{SKIP}_seed{SEED}{'_4bit' if QUANTIZED else ''}.jsonl")

extra2_grading_results = []
if os.path.exists(extra2_grading_path):
    with open(extra2_grading_path) as f:
        extra2_grading_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(extra2_grading_results[:N_EXTRA]):
        item = extra_sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(extra2_grading_results):
        with open(extra2_grading_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(extra2_grading_results)} to {len(valid)} valid items.")
    extra2_grading_results = valid
    print(f"Resuming from {len(extra2_grading_results)} completed items")

if len(extra2_grading_results) >= N_EXTRA:
    print(f"All {N_EXTRA} items already done.")
else:
    print(f"Starting from item {len(extra2_grading_results) + 1}/{N_EXTRA} "
          f"({N_EXTRA - len(extra2_grading_results)} remaining)", flush=True)
    with tqdm(total=(N_EXTRA - len(extra2_grading_results)) * K_GRADING,
              desc="grading (extra round 2)", unit="sample") as pbar:
        for item_idx, item in enumerate(extra_sample):
            if item_idx < len(extra2_grading_results):
                continue
            _t0 = time.time()
            messages = pilot.prompts.build_grading_messages(item["image"])
            texts = generate_grading(messages, K_GRADING, TEMP)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "samples_raw": texts,
                "quantized": QUANTIZED,
                "elapsed_seconds": _elapsed,
            }
            extra2_grading_results.append(entry)
            with open(extra2_grading_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_GRADING)
            print(f"  item {item_idx + 1}/{N_EXTRA}: {_elapsed:.1f}s "
                  f"({len(extra2_grading_results)}/{N_EXTRA} done)", flush=True)

print(f"extra2_grading_results: {len(extra2_grading_results)} items")


In [ ]:
# Merge all THREE checkpoints (reference + notebook 06's extra + this
# notebook's extra) and re-run the stratified analysis at the full n=650.
#
# Precision-searches the reference and round-1 extra checkpoints the same
# way notebook 06 did (both precision suffixes), since a genuine mismatch
# should raise a clear error, not resolve to a "file not found" from
# guessing the wrong suffix.
import importlib
import json
import os

import pandas as pd

import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.entropy, pilot.plotting):
    importlib.reload(m)


def _find_checkpoint(base, label):
    candidates = {"bf16": f"{base}.jsonl", "4bit": f"{base}_4bit.jsonl"}
    found = {k: p for k, p in candidates.items() if os.path.exists(p)}
    if not found:
        raise AssertionError(
            f"No {label} checkpoint found at {candidates['bf16']} or "
            f"{candidates['4bit']}."
        )
    if len(found) > 1:
        raise AssertionError(
            f"Found {label} checkpoints under BOTH precisions: {list(found)}. "
            "Resolve manually before merging."
        )
    return next(iter(found.values()))


REFERENCE_CHECKPOINT = _find_checkpoint(
    f"{CHECKPOINT_DIR}/grading_7b_k{K_GRADING}_{model_slug}_n300_seed{SEED}",
    "reference",
)
EXTRA1_CHECKPOINT = _find_checkpoint(
    f"{CHECKPOINT_DIR}/grading_7b_extra_error_k{K_GRADING}_{model_slug}_n200_skip150_seed{SEED}",
    "round-1 extra",
)

with open(REFERENCE_CHECKPOINT) as f:
    reference_entries = [json.loads(line) for line in f if line.strip()]
assert len(reference_entries) == 300, (
    f"Expected 300 reference items, found {len(reference_entries)}."
)

with open(EXTRA1_CHECKPOINT) as f:
    extra1_entries = [json.loads(line) for line in f if line.strip()]
assert len(extra1_entries) == 200, (
    f"Expected 200 round-1 extra items, found {len(extra1_entries)}."
)

# Precision consistency across all three checkpoints -- checks EVERY entry
# in each list, not just entries[0]. Checking only the first entry of each
# file would miss a partially-mixed checkpoint (e.g. one that resumed under
# a different precision than it started with -- the generation cell's own
# resume logic does not currently guard against that, so this is the one
# place that would still catch it).
def _quantized_values(entries):
    return {bool(e["quantized"]) for e in entries}


ref_quantized = _quantized_values(reference_entries)
extra1_quantized = _quantized_values(extra1_entries)
extra2_quantized = _quantized_values(extra2_grading_results)
all_quantized = ref_quantized | extra1_quantized | extra2_quantized
if all_quantized != {QUANTIZED}:
    raise RuntimeError(
        f"Quantization mismatch: reference={ref_quantized}, "
        f"round-1 extra={extra1_quantized}, round-2 extra={extra2_quantized}, "
        f"this session={QUANTIZED}. Refusing to merge measurements taken "
        "under different precisions."
    )
print(f"Precision check OK: all three runs quantized={QUANTIZED}")


def score_entry(entry):
    digits = [pilot.parsing.parse_grading(t) for t in entry["samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    return {
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": entry["item"]["has_error"],
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "n_grading_parse_failures": sum(1 for d in digits if d is None),
        "majority_digit": majority,
        "parsed_digits": digits,
        "all_grading_samples_raw": entry["samples_raw"],
        "model_id": MODEL_ID,
        "quantized": entry["quantized"],
    }


reference_df = pd.DataFrame(score_entry(e) for e in reference_entries)
extra1_df = pd.DataFrame(score_entry(e) for e in extra1_entries)
extra2_df = pd.DataFrame(score_entry(e) for e in extra2_grading_results)

# Disjointness across all three pairs, not just reference-vs-each-extra.
key = lambda df: set(zip(df["orig_q"], df["pert_a"]))
overlap_r_e1 = key(reference_df) & key(extra1_df)
overlap_r_e2 = key(reference_df) & key(extra2_df)
overlap_e1_e2 = key(extra1_df) & key(extra2_df)
assert not overlap_r_e1, f"{len(overlap_r_e1)} items overlap between reference and round-1 extra"
assert not overlap_r_e2, f"{len(overlap_r_e2)} items overlap between reference and round-2 extra"
assert not overlap_e1_e2, f"{len(overlap_e1_e2)} items overlap between round-1 and round-2 extra"

combined = pd.concat([reference_df, extra1_df, extra2_df], ignore_index=True)
print(f"Combined: {len(reference_df)} reference + {len(extra1_df)} extra(round 1) + "
      f"{len(extra2_df)} extra(round 2) = {len(combined)} total")

gt = combined["has_error"].astype(bool)
print(f"  has_error=1: {int(gt.sum())} items, {int((~combined.loc[gt,'grading_correct']).sum())} misgraded")
print(f"  has_error=0: {int((~gt).sum())} items, {int((~combined.loc[~gt,'grading_correct']).sum())} misgraded")

print()
print("=" * 70)
print("STRATIFIED ANALYSIS (combined, 7B, n=650 has_error=1)")
print("=" * 70)
out = pilot.plotting.stratified_auroc(
    combined, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in out["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {out['sign_reversal']}")
print(f"  pooled_understates : {out['pooled_understates']}")

print()
print("=" * 70)
print("VERDICT (reusing the registered 0.70 threshold, not a new one)")
print("=" * 70)
error_stratum = out["strata"][True]
minority = min(error_stratum["n_error"], error_stratum["n_correct"])
min_n = pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
threshold = pilot.plotting.SCALEUP_PREREGISTRATION["reasoning_stratum_auroc_min"]

if minority < min_n:
    print(f"  Still underpowered ({minority} < {min_n}).")
elif error_stratum["auroc"] >= threshold and error_stratum["excludes_chance"]:
    print(f"  CONFIRMED: AUROC {error_stratum['auroc']:.3f} clears the registered "
          f"{threshold} threshold with adequate power ({minority} >= {min_n}).")
else:
    print(f"  NOT CONFIRMED: adequately powered ({minority} >= {min_n}) but AUROC "
          f"{error_stratum['auroc']:.3f} does not clear {threshold}, or its CI includes chance.")

print()
print("=" * 70)
print("MATCHED-n MODEL COMPARISON (this is the whole point of this notebook)")
print("=" * 70)
print(f"  7B has_error=1, n=650 : {error_stratum['auroc']:.3f} "
      f"[{error_stratum['ci_low']:.3f}, {error_stratum['ci_high']:.3f}]  n_wrong={error_stratum['n_error']}")
print("  3B has_error=1, n=650 : 0.854 [0.796, 0.902]  n_wrong=42  (report S7.5)")
print("  Both model sizes now reported on the identical 650 has_error=1 items")
print("  (same seed=42 ordering both draws used).")


In [ ]:
# Save cell: the combined CSV, Drive first then repo + push. Identical
# pattern to notebook 06's save cell (git identity configured explicitly --
# a fresh Colab runtime has no git user.name/email set, which failed
# notebook 08's push with "Author identity unknown" rather than the usual
# 403; configuring it here avoids that specific failure mode, though the
# push itself may still 403 as it always has in this project).
import subprocess
from datetime import datetime, timezone
from getpass import getpass

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
csv_name = (f"grading_7b_matched_n650_"
            f"{'4bit_' if QUANTIZED else ''}{model_slug.lower()}_{timestamp}.csv")

drive_results = "/content/drive/MyDrive/uncertainty-math-vlm/results"
os.makedirs(drive_results, exist_ok=True)
combined.to_csv(f"{drive_results}/{csv_name}", index=False)
print(f"Backup written to {drive_results}/{csv_name}")

os.makedirs("repo/results", exist_ok=True)
csv_path = f"repo/results/{csv_name}"
combined.to_csv(csv_path, index=False)
print(f"Wrote {csv_path} ({len(combined)} rows)")

_REDACT = []


def git(*args):
    result = subprocess.run(["git", "-C", "repo", *args], capture_output=True, text=True)
    output = (result.stdout or "") + (result.stderr or "")
    for secret in _REDACT:
        if secret:
            output = output.replace(secret, "***")
    if result.returncode != 0 and output.strip():
        print(output.strip())
    return result


git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{csv_name}")
commit = git("commit", "-m", f"Add 7B matched-n650 grading results: {csv_name}")
if commit.returncode != 0:
    print("git commit failed (see above) -- CSV is safe on Drive.")

GH_PUSH_TOKEN = (globals().get("GH_TOKEN") or "").strip()
if not GH_PUSH_TOKEN:
    GH_PUSH_TOKEN = getpass("GitHub token (to push results), then press Enter: ").strip()
_REDACT.append(GH_PUSH_TOKEN)

if not GH_PUSH_TOKEN:
    print("No token given -- skipping push. CSV is saved on Drive and in repo/results/.")
else:
    push_url = REPO_URL.replace("https://", f"https://{GH_PUSH_TOKEN}@")
    if git("fetch", push_url, "main").returncode == 0:
        if git("rebase", "FETCH_HEAD").returncode != 0:
            git("rebase", "--abort")
            print("Rebase onto remote failed; attempting push anyway.")
    if git("push", push_url, "HEAD:main").returncode == 0:
        print("Pushed results.")
    else:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")
